条件分支示例


In [ ]:
# 导入类型定义模块
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

# 定义天气状态数据结构
class WeatherState(TypedDict):
    temperature: int        # 温度值
    recommendation: str     # 穿衣建议

def check_temperature(state: WeatherState) -> dict:
    """获取温度节点"""
    # 这里可以调用真实的天气 API
    return {"temperature": 25}

def route_by_temperature(state: WeatherState) -> Literal["cold", "warm", "hot"]:
    """根据温度路由（条件分支的核心逻辑）"""
    temp = state["temperature"]
    if temp < 15:
        return "cold"       # 寒冷：跳转到 cold 节点
    elif temp < 25:
        return "warm"       # 温暖：跳转到 warm 节点
    else:
        return "hot"        # 炎热：跳转到 hot 节点

# 三个不同温度范围的建议节点
def recommend_cold(state: WeatherState) -> dict:
    return {"recommendation": "穿厚外套！"}

def recommend_warm(state: WeatherState) -> dict:
    return {"recommendation": "穿长袖就好。"}

def recommend_hot(state: WeatherState) -> dict:
    return {"recommendation": "穿短袖，记得防晒！"}

# 构建图
graph = StateGraph(WeatherState)

# 添加所有节点
graph.add_node("check", check_temperature)
graph.add_node("cold", recommend_cold)
graph.add_node("warm", recommend_warm)
graph.add_node("hot", recommend_hot)

# 设置起始边：从 START 到 check 节点
graph.add_edge(START, "check")

# 添加条件边：根据 route_by_temperature 的返回值决定下一个节点
graph.add_conditional_edges(
    "check",                  # 源节点
    route_by_temperature,     # 路由函数
    {
        "cold": "cold",       # 路由映射：返回值 -> 目标节点
        "warm": "warm",
        "hot": "hot",
    }
)

# 所有建议节点都指向 END（结束）
graph.add_edge("cold", END)
graph.add_edge("warm", END)
graph.add_edge("hot", END)

# 编译并运行图
app = graph.compile()

# 测试：传入空状态，由 check_temperature 节点生成温度值
result = app.invoke({})
print(f"温度：{result['temperature']}° C")
print(f"建议：{result['recommendation']}")

# 可视化图结构
from IPython.display import Image, display

# 使用 Graphviz 渲染 (Colab 最稳定的方案)
try:
    display(Image(app.get_graph(xray=True).draw_png()))
except Exception as e:
    # 备用方案：使用 Mermaid 文本格式
    print(f"Graphviz 渲染失败: {e}")
    print("\n使用 Mermaid 文本方式显示:")
    print(app.get_graph(xray=True).draw_mermaid())